<a href="https://colab.research.google.com/github/dhaneshguptha15253-tech/Fraud-detection-agent/blob/main/fraud_detection_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [48]:
!pip install -q langchain langchain-groq

In [49]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [51]:
from langchain.chat_models import init_chat_model

# Initialize using the currently active and supported Llama 3.1 8B model on Groq
llm = init_chat_model("openai/gpt-oss-20b", model_provider="groq")

In [ ]:
from langchain.agents import create_agent

system_prompt = """You are a fraud detection assistant.
Help the user decide whether a transaction looks risky and explain your reasoning clearly."""

agent = create_agent(
    model=llm,
    tools=[],  # no tools attached yet
    system_prompt=system_prompt
)

print("Agent ready — no tools attached.")

In [ ]:
!pip install -q --upgrade torchao

In [ ]:
!pip install -q transformers peft

In [ ]:
from transformers import pipeline

# Using a small, lightweight model to test your pipeline
repo_name = "facebook/opt-125m"

generator = pipeline("text-generation", model=repo_name, device="cuda")

In [ ]:
from langchain.tools import tool

@tool
def analyze_transaction(
    transaction_type: str,
    amount: float,
    sender_balance_before: float,
    sender_balance_after: float,
    recipient_balance_before: float,
    recipient_balance_after: float
) -> str:
    """Analyze a financial transaction for fraud risk.
    Returns risk level (HIGH/LOW) with reasoning."""

    prompt = f"""Analyze this transaction for fraud risk:
- Type: {transaction_type}
- Amount: ${amount:,.2f}
- Sender Balance Before: ${sender_balance_before:,.2f}
- Sender Balance After: ${sender_balance_after:,.2f}
- Recipient Balance Before: ${recipient_balance_before:,.2f}
- Recipient Balance After: ${recipient_balance_after:,.2f}"""

    messages = [{"role": "user", "content": prompt}]
    output = generator(messages, max_new_tokens=150, return_full_text=False)[0]

    return output["generated_text"]

In [ ]:
system_prompt = """You are a fraud detection assistant.

When asked about a transaction, use the analyze_transaction tool to check for fraud risk.

Based on the analysis:
- If HIGH risk: Recommend blocking the transaction
- If LOW risk: Recommend approving the transaction

Always explain your reasoning clearly."""

# Re-initialize the agent to bind the newly updated LLM definition
agent = create_agent(
    model=llm,
    tools=[analyze_transaction],
    system_prompt=system_prompt,
)

print("Agent ready with the fraud tool attached and updated model llama3-8b-8192.")

In [ ]:
response = agent.invoke({
    "messages": [{"role": "user", "content": """Analyze this:
- Type: PAYMENT
- Amount: $150
- Sender balance: $5,000 before, $4,850 after
- Recipient balance: $1,000 before, $1,150 after"""}]
})

print(response["messages"][-1].content)